# MLP cross-check — a plain neural net on the GNN inputs, no message passing (HI-Small)

- **Question:** the GBT cross-check (§7.7) showed PNA and the Transformer above LightGBM on the same inputs. Is that message passing, or just "a neural net"? An MLP on exactly the same 93 columns — the readout's own input `[e_seed ‖ x_src ‖ x_dst]` without the learned node states — answers that.
- **Inputs:** 81 edge features (20 baseline + 61 GFP V0, as stored in the graph files) + the 6 + 6 entity one-hots of the two endpoints; one feature set only (`full`). The 81 edge columns are standardised on the train rows (std floor 1e-3); the one-hots stay 0/1.
- **Knobs:** none per run. The recipe is chosen once by random search (100 unique trials, one seed each; trial 0 = funnel 93 → 512 → 256 → 128 → 1, BatchNorm, dropout 0.2) on the **train rows**, scored on **validation**; then four seeds (42–45) are trained with it. Test is scored once per seed; every metric on train / val / test. F1 at the paper's 0.5 threshold is stored alongside for comparison only.
- **Recipe (`mlp_core.py`):** three shapes (flat, funnel, resnet), Adam at a constant LR, `BCEWithLogitsLoss(pos_weight)`, cap 60 epochs with **early stopping (patience 10) on validation PR-AUC** — every run keeps its best-validation-PR-AUC epoch, the threshold is swept on validation there, and trials are ranked by validation PR-AUC (threshold-free).
- **Everything shared** comes from `gnn_core.py` (threshold sweep, metrics, loss diagnostic, curves, summaries) and `gbt_core.py` (the 93-column matrix, predictions file) through `mlp_core.py`; this notebook only sets the budget and the paths. Logged search fits and finished seeds are skipped on rerun, so the search can span Kaggle sessions when `Outputs/MLP/search/` is committed in between.

## 0. Environment (Kaggle)

- Pinned PyTorch / PyG stack known to work on Kaggle GPU — do not change versions or order (PyG is needed to read the graph files).
- `lightgbm` is imported by `gbt_core` (the matrix builder); Kaggle ships it, the install line is a no-op then.
- Then clone (or pull) the public repo so the committed `gnn_core.py` / `gbt_core.py` / `mlp_core.py` are the ones imported.

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric
!pip install -q lightgbm

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/sandrokhizanishvili/gnn_for_fraudulent_patterns.git'
REPO_DIR = '/kaggle/working/gnn_for_fraudulent_patterns'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline
sys.path.append(REPO_DIR)

## 1. Configuration

- The only settings cell: the search budget, the seeds and the paths.
- `OUT_ROOT` sits inside the repo clone, mirroring `Outputs/MLP/`: committed search fits (`search/trials.csv`) and finished seeds are skipped on the next session.

In [ ]:
import json

import gnn_core as core
import gbt_core
import mlp_core

N_TRIALS     = mlp_core.N_TRIALS       # 100 unique trials; trial 0 = the default knobs (funnel 93 -> 512 -> 256 -> 128 -> 1)
SEARCH_SEEDS = mlp_core.SEARCH_SEEDS   # (42,): one seed per trial; the trial score is its best val PR-AUC
SEEDS        = mlp_core.SEEDS          # (42, 43, 44, 45) final seeds; 42 is the run compared with the GNN runs

DATA_DIR = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn'
OUT_ROOT = f'{REPO_DIR}/Outputs/MLP'
os.makedirs(f'{OUT_ROOT}/search', exist_ok=True)
print(f'torch {core.torch.__version__} | device {core.device} | cap {mlp_core.EPOCHS} epochs, patience {mlp_core.PATIENCE} '
      f'on validation PR-AUC | out {OUT_ROOT}')
print('trial 0:', json.dumps(mlp_core.DEFAULT_KNOBS))
print('search space:', json.dumps(mlp_core.SEARCH_SPACE))

## 2. Load data

- One row per edge from the graph files: `edge_attr` (81) ‖ `x[src]` (6) ‖ `x[dst]` (6); labels of the train / val windows from their own snapshots (the GBT matrix, asserted against `Progress_Report.md` §4).
- The 81 edge columns are standardised with the train-row mean / std; the 12 one-hots stay 0/1. The whole matrix then goes to the GPU once.

In [ ]:
data = gbt_core.load_gbt_data(DATA_DIR)
scaler = mlp_core.standardise(data)
X, y = mlp_core.to_tensors(data)

## 3. Model

- `mlp_core.MLP`: flat / funnel = `[Linear → BatchNorm → ReLU → Dropout]` per hidden width, then `Linear → 1`; resnet = `Linear → hidden`, residual blocks `z + Linear(Dropout(ReLU(BatchNorm(Linear(z)))))`, then `Linear → 1`.
- Trial 0 below; the search draws the shape, widths, depth, normalisation, dropout, learning rate, weight decay, `pos_weight` and batch size.

In [ ]:
model = mlp_core.MLP(X.shape[1], mlp_core.DEFAULT_KNOBS)
print(model)
print(f"trial 0: {mlp_core.DEFAULT_KNOBS['shape']} {mlp_core.widths_text(X.shape[1], model.widths, mlp_core.DEFAULT_KNOBS['shape'])} "
      f"| params {core.count_params(model):,}")
del model

## 4. Hyperparameter search — random search on the train rows, early stopping on validation PR-AUC

- Every trial is trained once (seed 42); the fit stops after 10 epochs without a better validation PR-AUC and keeps its best epoch; that epoch's validation PR-AUC ranks the trial (F1 and threshold recorded).
- `search/trials.csv` gets one line per (trial, seed) as soon as it finishes (shape, widths, params, epochs, seconds per epoch); `search/best_params.json` holds the winner.
- Test is not touched here.

In [ ]:
best_knobs, trials = mlp_core.random_search(data, X, y, f'{OUT_ROOT}/search', N_TRIALS, SEARCH_SEEDS)
scores = trials.groupby('trial').agg(shape=('shape', 'first'), widths=('widths', 'first'), params=('params', 'first'),
                                     val_pr_auc=('val_pr_auc', 'mean'), val_f1=('val_f1', 'mean'),
                                     best_epoch=('best_epoch', 'mean'), sec_per_epoch=('sec_per_epoch', 'mean'), n_seeds=('seed', 'count'))
scores[scores['n_seeds'] == len(SEARCH_SEEDS)].sort_values('val_pr_auc', ascending=False).head(10).round(4)

## 5. Final runs — four seeds with the chosen knobs

- One call per seed: train rows → early stopping on validation PR-AUC → the best epoch kept, threshold swept there → all splits scored → `results.json`, `history.csv`, `curves.png`, `best.pt`, `predictions.csv` under `OUT_ROOT/<run>/`.
- `history.csv` has one row per epoch; `curves.png` marks the kept epoch.

In [ ]:
rows = []
for seed in SEEDS:
    print('=' * 90)
    rows.append(mlp_core.train_final(best_knobs, data, X, y, seed, OUT_ROOT))
print('=' * 90)
print('batch complete')

## 6. Comparison across the seeds

- `batch_summary_mlp.csv` holds every `<split>_<metric>` column plus `threshold`, `best_epoch`, `params`, `invariant_params` (0 by convention).
- `seed_summary_mlp.csv`: mean ± std over the four seeds. Only test F1 and PR-AUC are fair scores here — validation picked the hyperparameters, the kept epoch and the threshold.

In [ ]:
summary = core.summarize_batch(rows, OUT_ROOT, 'batch_summary_mlp.csv')
display(summary[core.DISPLAY_COLS].round(4))
seeds = mlp_core.seed_summary(rows, OUT_ROOT)
seeds[['n_seeds', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std',
       'test_pr_auc_mean', 'test_pr_auc_std', 'best_epoch_mean']].round(4)

## 7. Download

- Zip `OUT_ROOT`; extract into `Outputs/MLP/` in the repo and commit the small result files (`best.pt` and `predictions.csv` stay out of git). Commit `search/trials.csv` after a session that did not finish the search — the next session resumes from it.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/Outputs_MLP', 'zip', OUT_ROOT)